# 00 Data Understanding

Mục tiêu notebook:

- Tải dữ liệu thô từ thư mục `data/raw`.

- Kiểm tra schema, kiểu dữ liệu, số dòng, phạm vi thời gian.

- Đánh giá tần suất quan sát và phát hiện các mốc thời gian bị khuyết.

- Xác định dải thời gian giao nhau dùng chung cho 3 biến NASDAQ100, CPIAUCSL, FEDFUNDS.

In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

ROOT = Path.cwd().resolve().parent
RAW_DIR = ROOT / "data" / "raw"

files = {
    "NASDAQ100": RAW_DIR / "NASDAQ100.csv",
    "CPIAUCSL": RAW_DIR / "CPIAUCSL.csv",
    "FEDFUNDS": RAW_DIR / "FEDFUNDS.csv",
}

for name, path in files.items():
    print(f"{name}: {path} | exists={path.exists()}")

NASDAQ100: S:\vscode\nasdaq-macro-shocks-analysis\data\raw\NASDAQ100.csv | exists=True
CPIAUCSL: S:\vscode\nasdaq-macro-shocks-analysis\data\raw\CPIAUCSL.csv | exists=True
FEDFUNDS: S:\vscode\nasdaq-macro-shocks-analysis\data\raw\FEDFUNDS.csv | exists=True


In [7]:
def load_series(csv_path: Path, value_name: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df = df.rename(columns={"observation_date": "date", value_name: "value"})
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.sort_values("date").reset_index(drop=True)
    return df

datasets = {name: load_series(path, name) for name, path in files.items()}

for name, df in datasets.items():
    print("=" * 80)
    print(name)
    print(df.info())
    print(df.head(3))
    print(df.tail(3))
    print()

NASDAQ100
<class 'pandas.DataFrame'>
RangeIndex: 10530 entries, 0 to 10529
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    10530 non-null  datetime64[us]
 1   value   10171 non-null  float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 164.7 KB
None
        date   value
0 1986-01-02  131.25
1 1986-01-03  130.55
2 1986-01-06  130.35
            date     value
10527 2026-05-11  29320.66
10528 2026-05-12  29064.80
10529 2026-05-13  29366.94

CPIAUCSL
<class 'pandas.DataFrame'>
RangeIndex: 952 entries, 0 to 951
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    952 non-null    datetime64[us]
 1   value   951 non-null    float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 15.0 KB
None
        date  value
0 1947-01-01  21.48
1 1947-02-01  21.62
2 1947-03-01  22.00
          date    value
949 2026-02

In [8]:
def infer_freq_safe(idx: pd.DatetimeIndex) -> str:
    if len(idx) < 3:
        return "insufficient_obs"
    freq = pd.infer_freq(idx)
    return freq if freq is not None else "irregular"

def missing_periods(df: pd.DataFrame, expected_freq: str) -> pd.DatetimeIndex:
    clean = df.dropna(subset=["date"]).copy()
    clean = clean.drop_duplicates(subset=["date"]).sort_values("date")
    full_idx = pd.date_range(clean["date"].min(), clean["date"].max(), freq=expected_freq)
    return full_idx.difference(pd.DatetimeIndex(clean["date"]))

profile_rows = []

for name, df in datasets.items():
    s = df.dropna(subset=["date"]).drop_duplicates(subset=["date"]).sort_values("date")
    idx = pd.DatetimeIndex(s["date"])
    inferred = infer_freq_safe(idx)
    expected_freq = "B" if name == "NASDAQ100" else "MS"
    missing = missing_periods(s, expected_freq)
    profile_rows.append(
        {
            "series": name,
            "rows_total": len(df),
            "rows_non_null_value": int(df["value"].notna().sum()),
            "start_date": s["date"].min(),
            "end_date": s["date"].max(),
            "inferred_freq": inferred,
            "expected_freq_check": expected_freq,
            "missing_periods_count": len(missing),
        }
    )

    print(f"{name} | inferred={inferred} | expected={expected_freq} | missing_count={len(missing)}")
    if len(missing) > 0:
        print("  first_missing:", [d.strftime("%Y-%m-%d") for d in missing[:10]])

profile_df = pd.DataFrame(profile_rows)
profile_df

NASDAQ100 | inferred=B | expected=B | missing_count=0
CPIAUCSL | inferred=MS | expected=MS | missing_count=0
FEDFUNDS | inferred=MS | expected=MS | missing_count=0


,series,rows_total,rows_non_null_value,start_date,end_date,inferred_freq,expected_freq_check,missing_periods_count
0,NASDAQ100,10530,10171,1986-01-02,2026-05-13,B,B,0
1,CPIAUCSL,952,951,1947-01-01,2026-04-01,MS,MS,0
2,FEDFUNDS,862,862,1954-07-01,2026-04-01,MS,MS,0


In [9]:
common_start = max(df["date"].min() for df in datasets.values())
common_end = min(df["date"].max() for df in datasets.values())

print("Common usable date range:")
print("- start:", common_start.date())
print("- end  :", common_end.date())

for name, df in datasets.items():
    within = df[(df["date"] >= common_start) & (df["date"] <= common_end)]
    print(f"{name}: {len(within):,} observations in common range")

Common usable date range:
- start: 1986-01-02
- end  : 2026-04-01
NASDAQ100: 10,500 observations in common range
CPIAUCSL: 483 observations in common range
FEDFUNDS: 483 observations in common range


In [10]:
summary_path = ROOT / "outputs" / "tables" / "data_understanding_summary.csv"
summary_path.parent.mkdir(parents=True, exist_ok=True)
profile_df.to_csv(summary_path, index=False)
summary_path

WindowsPath('S:/vscode/nasdaq-macro-shocks-analysis/outputs/tables/data_understanding_summary.csv')